In [18]:
import sys
sys.path.append("..")

import json
import torch
import pandas as pd
import optuna

from src.data import create_dataloader
from src.models import build_mlp
from src.train import train_model

def load_and_predict(model_path, config, test_df, feature_cols, device):
    """
    model_path: path to the saved .pt model file
    config: dict with hyperparameters (hidden_layers, dropout, lr, batch_size, grad_clip)
    test_df: test dataframe
    feature_cols: list of feature column names
    device: 'cpu' or 'cuda'
    """

    # 1. Build the model with the same architecture used during training
    model = build_mlp(
        config={"hidden_layers": config["hidden_layers"], "dropout": config["dropout"]},
        input_dim=len(feature_cols),
        output_dim=train_df[target_col].nunique()  # number of classes
    ).to(device)

    # 2. Load the trained weights
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()  # switch to evaluation mode (disables dropout)

    # 3. Prepare input tensor
    X = torch.tensor(test_df[feature_cols].values, dtype=torch.float32).to(device)

    # 4. Forward pass to get predictions
    with torch.no_grad():
        logits = model(X)
        probs = torch.softmax(logits, dim=1).cpu().numpy()  # convert to probabilities

    return probs


In [19]:
import numpy as np

def ensemble_predict(all_models, top_configs, test_df, feature_cols, device):
    """
    all_models: dict {config_id: [list of model paths]}
    top_configs: list of 3 hyperparameter dicts (top Optuna configs)
    test_df: test dataframe
    feature_cols: list of feature names
    device: 'cpu' or 'cuda'
    """

    all_probs = []  # will store predictions from all 15 models

    for config_id, model_paths in all_models.items():
        config = top_configs[config_id - 1]  # match config_id (1,2,3) to index (0,1,2)

        for model_path in model_paths:
            probs = load_and_predict(
                model_path=model_path,
                config=config,
                test_df=test_df,
                feature_cols=feature_cols,
                device=device
            )
            all_probs.append(probs)

    # Convert list of (N, num_classes) arrays into shape (15, N, num_classes)
    all_probs = np.stack(all_probs, axis=0)

    # Ensemble = mean over models
    ensemble_probs = np.mean(all_probs, axis=0)

    return ensemble_probs, all_probs  # return both mean and raw predictions


In [20]:
import numpy as np

def compute_uncertainty(all_probs):
    """
    all_probs: numpy array of shape (num_models, N, num_classes)
               predictions from all 15 models

    Returns:
        variance_uncertainty: (N,) variance across models
        entropy_uncertainty:  (N,) entropy of ensemble probabilities
    """

    # 1. Variance across models (model disagreement)
    # Compute variance for each sample across the 15 models
    variance_uncertainty = np.var(all_probs, axis=0).mean(axis=1)
    # shape: (N, num_classes) -> mean over classes -> (N,)

    # 2. Entropy of the mean probability distribution
    # First compute ensemble mean
    ensemble_probs = np.mean(all_probs, axis=0)  # shape (N, num_classes)

    # Add small epsilon to avoid log(0)
    eps = 1e-12
    entropy_uncertainty = -np.sum(ensemble_probs * np.log(ensemble_probs + eps), axis=1)
    # shape: (N,)

    return variance_uncertainty, entropy_uncertainty


In [21]:
import pandas as pd
import numpy as np

def save_final_predictions(test_df, ensemble_probs, variance_uncertainty, entropy_uncertainty, output_path="final_predictions.csv"):
    """
    test_df: original test dataframe
    ensemble_probs: (N, num_classes) mean probabilities from ensemble
    variance_uncertainty: (N,) variance-based uncertainty
    entropy_uncertainty: (N,) entropy-based uncertainty
    output_path: where to save the final CSV
    """

    # 1. Final predicted class (argmax over ensemble probabilities)
    final_preds = np.argmax(ensemble_probs, axis=1)

    # 2. Build output dataframe
    out_df = pd.DataFrame({
        "prediction": final_preds,
        "uncertainty_variance": variance_uncertainty,
        "uncertainty_entropy": entropy_uncertainty
    })

    # 3. Save to CSV
    out_df.to_csv(output_path, index=False)

    return out_df


In [22]:
import os
from collections import defaultdict

def load_all_models_from_folder(models_dir="models"):
    """
    Scans the models/ directory and reconstructs the all_models dict.
    Expected filenames: config_{id}_fold_{k}.pt
    """

    all_models = defaultdict(list)

    for fname in os.listdir(models_dir):
        if fname.endswith(".pt") and fname.startswith("config_"):
            # Example filename: config_1_fold_3.pt
            parts = fname.replace(".pt", "").split("_")
            config_id = int(parts[1])
            fold_id = int(parts[3])

            full_path = os.path.join(models_dir, fname)
            all_models[config_id].append(full_path)

    # Sort folds inside each config (0..4)
    for cfg in all_models:
        all_models[cfg] = sorted(
            all_models[cfg],
            key=lambda x: int(x.split("_fold_")[1].replace(".pt", ""))
        )

    return dict(all_models)

# Build the dictionary
all_models = load_all_models_from_folder("models")

all_models


{3: ['models/config_3_fold_0.pt',
  'models/config_3_fold_1.pt',
  'models/config_3_fold_2.pt',
  'models/config_3_fold_3.pt',
  'models/config_3_fold_4.pt'],
 2: ['models/config_2_fold_0.pt',
  'models/config_2_fold_1.pt',
  'models/config_2_fold_2.pt',
  'models/config_2_fold_3.pt',
  'models/config_2_fold_4.pt'],
 1: ['models/config_1_fold_0.pt',
  'models/config_1_fold_1.pt',
  'models/config_1_fold_2.pt',
  'models/config_1_fold_3.pt',
  'models/config_1_fold_4.pt']}

In [23]:
import json
from pathlib import Path

def load_top_configs(configs_dir="ensemble_configs"):
    """
    Loads the 3 best Optuna hyperparameter configs from JSON files.
    Returns a list of 3 dicts.
    """
    configs = []
    configs_dir = Path(configs_dir)

    for i in range(1, 4):
        cfg_path = configs_dir / f"top_{i}.json"
        with open(cfg_path, "r") as f:
            configs.append(json.load(f))

    return configs

# Load configs
top_configs = load_top_configs("ensemble_configs")

top_configs


[{'hidden_layers': [256, 128],
  'dropout': 0.006637638206590968,
  'lr': 0.004019188571880907,
  'batch_size': 64,
  'grad_clip': 4.785360125621526},
 {'hidden_layers': [256, 128],
  'dropout': 0.11371175255187554,
  'lr': 0.0016451365040683543,
  'batch_size': 64,
  'grad_clip': 4.992958230872855},
 {'hidden_layers': [256, 128],
  'dropout': 0.1146159188561183,
  'lr': 0.0015359149332646977,
  'batch_size': 64,
  'grad_clip': 4.979783467988117}]

In [24]:
import pandas as pd

# Load test dataset
test_df = pd.read_csv("../data/processed/test.csv")

# If you need train_df for output_dim or class count:
train_df = pd.read_csv("../data/processed/train.csv")
target_col = "class"
feature_cols = [c for c in train_df.columns if c != target_col]
device = "cuda" if torch.cuda.is_available() else "cpu"

In [26]:

ensemble_probs, all_probs = ensemble_predict(all_models, top_configs, test_df, feature_cols, device)
var_u, ent_u = compute_uncertainty(all_probs)
final_df = save_final_predictions(test_df, ensemble_probs, var_u, ent_u)


In [27]:
print("Ensemble predictions shape:", ensemble_probs.shape)
print("All model predictions shape:", all_probs.shape)
print("Variance uncertainty (first 10):", var_u[:10])
print("Entropy uncertainty (first 10):", ent_u[:10])

print("\nSample of final predictions:")
display(final_df.head(10))


Ensemble predictions shape: (3734, 26)
All model predictions shape: (15, 3734, 26)
Variance uncertainty (first 10): [1.8798599e-05 8.1727038e-08 1.4383180e-05 2.9553570e-05 4.0154369e-09
 7.3809083e-08 1.3856752e-10 6.8713632e-11 6.4119178e-08 6.6605042e-20]
Entropy uncertainty (first 10): [9.9685289e-02 3.5143730e-03 7.4031316e-02 9.0405397e-02 9.4004610e-04
 3.8605921e-03 2.5618947e-04 1.3996210e-04 8.0062272e-03 1.5912899e-08]

Sample of final predictions:


,prediction,uncertainty_variance,uncertainty_entropy
0,15,1.879860e-05,9.968529e-02
1,24,8.172704e-08,3.514373e-03
2,1,1.438318e-05,7.403132e-02
3,17,2.955357e-05,9.040540e-02
4,22,4.015437e-09,9.400461e-04
5,22,7.380908e-08,3.860592e-03
6,0,1.385675e-10,2.561895e-04
7,15,6.871363e-11,1.399621e-04
8,13,6.411918e-08,8.006227e-03
9,11,6.660504e-20,1.591290e-08


In [29]:
from sklearn.metrics import accuracy_score

# Prepare train features
X_train = train_df[feature_cols].values
y_train = train_df[target_col].values

# Convert to tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

# Collect predictions from all 15 models
train_probs_list = []

for config_id, model_paths in all_models.items():
    config = top_configs[config_id - 1]

    for model_path in model_paths:
        model = build_mlp(
            config={"hidden_layers": config["hidden_layers"], "dropout": config["dropout"]},
            input_dim=len(feature_cols),
            output_dim=train_df[target_col].nunique()
        ).to(device)

        state = torch.load(model_path, map_location=device)
        model.load_state_dict(state)
        model.eval()

        with torch.no_grad():
            logits = model(X_train_tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()

        train_probs_list.append(probs)

# Stack and average
train_probs = np.mean(np.stack(train_probs_list, axis=0), axis=0)

# Final predictions
train_preds = np.argmax(train_probs, axis=1)

# Accuracy
acc = accuracy_score(y_train, train_preds)
print("Ensemble accuracy on train_df:", acc)

from sklearn.metrics import accuracy_score

# True labels from test_df
y_test = test_df[target_col].values

# Ensemble predictions
y_pred = np.argmax(ensemble_probs, axis=1)

# Accuracy
acc_test = accuracy_score(y_test, y_pred)

print("True accuracy on test set:", acc_test)


Ensemble accuracy on train_df: 0.9939732142857143
True accuracy on test set: 0.9683985002678093


In [30]:
print("Ensemble predictions shape:", ensemble_probs.shape)
print("All model predictions shape:", all_probs.shape)
print("True accuracy on test set:", acc_test)

print("\nSample of final predictions:")
display(final_df.head(10))


Ensemble predictions shape: (3734, 26)
All model predictions shape: (15, 3734, 26)
True accuracy on test set: 0.9683985002678093

Sample of final predictions:


,prediction,uncertainty_variance,uncertainty_entropy
0,15,1.879860e-05,9.968529e-02
1,24,8.172704e-08,3.514373e-03
2,1,1.438318e-05,7.403132e-02
3,17,2.955357e-05,9.040540e-02
4,22,4.015437e-09,9.400461e-04
5,22,7.380908e-08,3.860592e-03
6,0,1.385675e-10,2.561895e-04
7,15,6.871363e-11,1.399621e-04
8,13,6.411918e-08,8.006227e-03
9,11,6.660504e-20,1.591290e-08
